Drop Views if it is exists

In [0]:
%sql
DROP VIEW IF EXISTS workspace.gold.dim_customers;

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.gold.dim_customers AS
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key, -- Surrogate key
    ci.customer_id                          AS customer_id,
    ci.customer_key                         AS customer_number,
    ci.first_name                           AS first_name,
    ci.last_name                            AS last_name,
    la.country                              AS country,
    ci.marital_status                       AS marital_status,
    CASE 
        WHEN ci.gender != 'n/a' THEN ci.gender -- CRM is the primary source for gender
        ELSE COALESCE(ca.gender, 'n/a')  			   -- Fallback to ERP data
    END                                AS gender,
    ca.birthdate                       AS birthdate,
    ci.created_date                    AS create_date
FROM silver.crm_customers ci
LEFT JOIN silver.erp_cust_info ca
    ON ci.customer_key = ca.customer_id
LEFT JOIN silver.erp_customer_location la
    ON ci.customer_key = la.company_id
where ci.customer_id IS NOT NULL
LIMIT 100;

In [0]:
%sql
select * from workspace.gold.dim_customers limit 100 

Current catlog product in market

In [0]:
%sql
DROP VIEW IF EXISTS workspace.gold.dim_products;

CREATE or REPLACE VIEW gold.dim_products AS
SELECT
    ROW_NUMBER() OVER (ORDER BY pn.start_date, pn.product_key) AS product_key, -- Surrogate key
    pn.product_id       AS product_id,
    pn.product_key      AS product_number,
    pn.product_name     AS product_name,
    pn.category_id      AS category_id,
    pc.category         AS category,
    pc.subcategory      AS subcategory,
    pc.maintenance_flag AS maintenance,
    pn.product_cost     AS cost,
    pn.product_line     AS product_line,
    pn.start_date AS start_date
FROM silver.crm_products pn
LEFT JOIN silver.erp_product_category pc
    ON pn.category_id = pc.category_id
WHERE pn.end_date IS NULL;

In [0]:
%sql
select * from workspace.gold.dim_products limit 100

Created fact Table

In [0]:
%sql
DROP VIEW IF EXISTS workspace.gold.fact_sales;

CREATE OR REPLACE VIEW gold.fact_sales AS
SELECT
    sd.order_number   AS order_number,
    pr.product_key    AS product_key,
    cu.customer_key   AS customer_key,
    sd.order_date     AS order_date,
    sd.ship_date      AS shipping_date,
    sd.due_date       AS due_date,
    sd.sales_amount   AS sales_amount,
    sd.quantity       AS quantity,
    sd.price          AS price
FROM silver.crm_sales sd
LEFT JOIN gold.dim_products pr
    ON sd.product_key = pr.product_number
LEFT JOIN gold.dim_customers cu
    ON sd.customer_id = cu.customer_id
WHERE cu.customer_key IS NOT NULL;

In [0]:
%sql 
select * from workspace.gold.fact_sales limit 100 